# Remaining GPU work for the paper revision

Runs everything still needed on the GPU, all **scoring only (no training)**, through `revision_gpu_jobs.py`:

| Stage | Jobs | What | GPU time |
|---|---|---|---|
| RetiZero Messidor-2 | 7 | matched t42/43/44 (CLAHE off), CLAHE t42/43/44, saved app checkpoint | about 15 min |
| Qwen Messidor-2 | 14 | r16 CLAHE-on lr 2e-4 ×3, r16 CLAHE-on lr 5e-4 ×3, r16 CLAHE-off lr 5e-4 ×3, r16 CLAHE-off lr 2e-4 ×3, zero-shot on/off | about 7 h |
| Qwen timing | 7 | warm per-image timing, generation check and power for the 7 configurations without a timing record | about 2 h |

Verified Messidor-2 = all 1,744 gradable images with adjudicated grades, 874 exams, preprocessed straight from the raw photos.

**Before you start**

1. The fixes in `docs/handoffs/codex_revision_runner_fixes.md` are on branch `qwen-rank-lr2e-4`, and that branch is checked out or merged in **`D:\Afterimage_Reproduction`**. The runner finds `D:\Afterimage_Experiments` from the checkout's location, so it must not run from a worktree. Cell 2 checks all of this and stops if anything is missing.
2. Kernel: **Python (afterimage_env)**, with scipy installed.
3. Nothing else is using the GPU.

**Stopping and resuming.** Interrupt the kernel (■). The running job is killed and the next run of the same cell starts it again; finished jobs are always skipped. Every step's full console output is saved in `D:\Afterimage_Experiments\final_runs\final_02\revision_notebook_logs\`.

## 1. Setup

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import os
import subprocess
import sys

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current: {sys.executable}")

REPO = Path(r"D:\Afterimage_Reproduction")
EXPERIMENTS = Path(r"D:\Afterimage_Experiments")
MESSIDOR_PACKAGE = Path(r"D:\Afterimage_data_share\Messidor-2")
RUNNER = REPO / "finetuning" / "revision_jobs" / "revision_gpu_jobs.py"
ANALYSIS = REPO / "finetuning" / "paper_tables" / "build_revision_analysis.py"
FINAL_02 = EXPERIMENTS / "final_runs" / "final_02"
LOGS = FINAL_02 / "revision_notebook_logs"
SNAPSHOT = LOGS / "protected_files_before.json"
PYTHON = sys.executable
LOGS.mkdir(parents=True, exist_ok=True)


def run_step(*arguments, name=None):
    """Run the runner (or any script) and stream its output here and into a log file. Returns the exit code."""
    name = name or arguments[0]
    log = LOGS / f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{name}.log"
    command = [PYTHON, "-B", *map(str, arguments)] if str(arguments[0]).endswith(".py") else \
              [PYTHON, "-B", str(RUNNER), *map(str, arguments)]
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    print(f"$ {' '.join(command)}\nlog: {log}\n", flush=True)
    with open(log, "w", encoding="utf-8") as stream:
        stream.write(f"$ {' '.join(command)}\nstarted {datetime.now(timezone.utc).isoformat()}\n\n")
        process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            code = process.wait()
        except KeyboardInterrupt:
            subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], capture_output=True)
            stream.write("\nInterrupted by the user; the process tree was stopped.\n")
            print("\nInterrupted. The running job was stopped; rerun this cell to resume.")
            raise
        stream.write(f"\nexit code {code}, ended {datetime.now(timezone.utc).isoformat()}\n")
    print(f"\nexit code {code}")
    return code


def load_runner():
    """Import revision_gpu_jobs.py from the checkout (CPU only: it builds the job list, starts nothing)."""
    spec = importlib.util.spec_from_file_location("revision_gpu_jobs", RUNNER)
    module = importlib.util.module_from_spec(spec)
    sys.dont_write_bytecode = True
    spec.loader.exec_module(module)
    return module


print("Python:", sys.executable)
print("Logs:", LOGS)

Python: C:\Users\moht2\.conda\envs\afterimage_env\python.exe
Logs: D:\Afterimage_Experiments\final_runs\final_02\revision_notebook_logs


## 2. Preflight

Reads files only. Stops with a clear message if:

- the checkout is not on the fixed branch
- the Messidor-2 package is incomplete
- the job list is not the expected 28

It also runs the CPU tests. Finally it records the size and modification time of every existing file in the earlier result folders, so cell 8 can prove none of them changed.

In [2]:
problems = []

# The checkout: right place, right code.
branch = subprocess.run(["git", "-C", str(REPO), "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain", "--", "finetuning"], capture_output=True, text=True).stdout.strip()
print(f"Checkout {REPO} on branch '{branch}'" + (" (uncommitted changes under finetuning/)" if dirty else ""))
if not RUNNER.is_file():
    raise RuntimeError(
        f"This checkout (branch '{branch}') does not have the revision runner yet, so nothing can run.\n"
        f"  1. Get the runner fixes onto branch qwen-rank-lr2e-4 (docs/handoffs/codex_revision_runner_fixes.md).\n"
        f"  2. Bring that branch into {REPO}, e.g.  git -C {REPO} merge qwen-rank-lr2e-4\n"
        f"  3. Restart the kernel and run from cell 1 again.")
else:
    runner_text, analysis_text = RUNNER.read_text(encoding="utf-8"), ANALYSIS.read_text(encoding="utf-8")
    fixes = {
        "1 energy file renamed": "revision_energy_per_job.csv" in runner_text and "revision_energy_per_job.csv" in analysis_text,
        "2 --rank-sweep removed": "rank_sweep" not in runner_text and "rank-sweep" not in runner_text,
        "4 CLAHE-off archived Messidor in the analysis": "messidor_archived_off" in analysis_text,
        "5 separate pilot folder": 'PILOT_TAG = "revision_pilot_01"' in runner_text,
    }
    for fix, ok in fixes.items():
        print(f"  [{'ok' if ok else 'MISSING'}] fix {fix}")
        if not ok:
            problems.append(f"fix {fix} is not in this checkout (see docs/handoffs/codex_revision_runner_fixes.md)")

# The Messidor-2 package.
raw = MESSIDOR_PACKAGE / "01_raw_images"
grades = MESSIDOR_PACKAGE / "02_adjudicated_grades" / "messidor_data.csv"
pairs = MESSIDOR_PACKAGE / "03_eye_pairing" / "pairs.csv"
n_raw = sum(1 for p in raw.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".tif")) if raw.is_dir() else 0
print(f"Messidor-2 package {MESSIDOR_PACKAGE}: {n_raw} raw images, grades {'ok' if grades.is_file() else 'MISSING'}, "
      f"pairs {'ok' if pairs.is_file() else 'MISSING'}")
if n_raw != 1748 or not grades.is_file() or not pairs.is_file():
    problems.append("the Messidor-2 package needs 1,748 raw images, messidor_data.csv and pairs.csv")

# The job list (fix 3 shows up here as the three r16 CLAHE-off lr 2e-4 jobs).
if not problems:
    runner = load_runner()
    jobs = runner.build_jobs()
    if runner.EXPERIMENTS.resolve() != EXPERIMENTS.resolve():
        problems.append(f"the runner resolves the experiments folder to {runner.EXPERIMENTS}, not {EXPERIMENTS}")
    by_kind = {}
    for queue, job in jobs:
        by_kind[f"{queue} {job.stage}"] = by_kind.get(f"{queue} {job.stage}", 0) + 1
    new_off = [job.name for _, job in jobs if job.name.startswith("r16_off_alpha16_lr0p0002_") and job.name.endswith("__messidor2_off")]
    print(f"\nJob list: {len(jobs)} jobs {by_kind}; r16 CLAHE-off lr 2e-4 Messidor-2 jobs: {len(new_off)}")
    expected = {"retizero messidor2": 7, "qwen messidor2": 14, "qwen timing": 7}
    if by_kind != expected or len(new_off) != 3:
        problems.append(f"expected {expected} with 3 r16 CLAHE-off lr 2e-4 jobs (fix 3), got {by_kind} and {len(new_off)}")

# CPU tests of the runner and the analysis.
for test in (REPO / "finetuning/revision_jobs/test_revision_gpu_jobs.py", REPO / "finetuning/paper_tables/test_revision_analysis.py"):
    result = subprocess.run([PYTHON, "-B", str(test)], cwd=REPO, capture_output=True, text=True)
    print(f"  [{'ok' if result.returncode == 0 else 'FAILED'}] {test.name}: {(result.stderr.strip().splitlines() or [''])[-1]}")
    if result.returncode != 0:
        problems.append(f"{test.name} failed:\n{result.stderr[-2000:]}")

if problems:
    raise RuntimeError("Preflight failed:\n- " + "\n- ".join(problems))

# Snapshot of the earlier results, which must stay untouched. Kept from the first run so a resume compares with the original state.
PROTECTED = [FINAL_02, EXPERIMENTS / "final_runs" / "final_02__pilot_01", EXPERIMENTS / "final_runs" / "final_01",
             EXPERIMENTS / "retizero_runs" / "retizero_01", EXPERIMENTS / "runs"]
if not SNAPSHOT.is_file():
    snapshot = {}
    for folder in PROTECTED:
        for path in folder.rglob("*"):
            if path.is_file() and LOGS not in path.parents:
                stat = path.stat()
                snapshot[str(path)] = [stat.st_size, stat.st_mtime_ns]
    SNAPSHOT.write_text(json.dumps(snapshot), encoding="utf-8")
    print(f"\nRecorded {len(snapshot):,} existing files in {SNAPSHOT.name}.")
else:
    print(f"\nUsing the snapshot from the first run: {SNAPSHOT}")
print("\nPreflight passed.")

Checkout D:\Afterimage_Reproduction on branch 'main' (uncommitted changes under finetuning/)
  [ok] fix 1 energy file renamed
  [ok] fix 2 --rank-sweep removed
  [ok] fix 4 CLAHE-off archived Messidor in the analysis
  [ok] fix 5 separate pilot folder
Messidor-2 package D:\Afterimage_data_share\Messidor-2: 1748 raw images, grades ok, pairs ok

Job list: 28 jobs {'retizero messidor2': 7, 'qwen messidor2': 14, 'qwen timing': 7}; r16 CLAHE-off lr 2e-4 Messidor-2 jobs: 3
  [ok] test_revision_gpu_jobs.py: OK
  [FAILED] test_revision_analysis.py: ModuleNotFoundError: No module named 'scipy'


RuntimeError: Preflight failed:
- test_revision_analysis.py failed:
Traceback (most recent call last):
  File "D:\Afterimage_Reproduction\finetuning\paper_tables\test_revision_analysis.py", line 15, in <module>
    import build_revision_analysis as bra  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Afterimage_Reproduction\finetuning\paper_tables\build_revision_analysis.py", line 36, in <module>
    from scipy import stats
ModuleNotFoundError: No module named 'scipy'


## 3. Check

The runner's own check: environment, Qwen base, GPU lock and disk space, then the planned jobs. It must end with **Check passed**. The Messidor-2 cohort shows as `later` until step 4 builds it.

In [ ]:
assert run_step("check") == 0, "Check failed: fix the items marked MISSING above, then rerun this cell."

## 4. Build the verified Messidor-2 cohort (CPU, about 5 minutes)

Writes four preprocessed copies to `D:\Afterimage_Experiments\datasets\messidor2_verified`: Qwen 512 px with CLAHE on and off, and RetiZero 224 px with CLAHE off and on. A rerun with the same inputs does nothing. The cell then checks the build against the known counts. The 1,057 images shared with the archived export must be pixel-identical to it.

In [ ]:
assert run_step("build-messidor", "--messidor-package", MESSIDOR_PACKAGE) == 0, "The cohort build failed; see the log above."

manifest = json.loads((EXPERIMENTS / "datasets/messidor2_verified/manifest.json").read_text(encoding="utf-8"))
archive = manifest.get("archive_check", {})
print(json.dumps({k: manifest[k] for k in ("images", "grade_counts", "exams", "parts", "opencv")}, indent=1))
print("archive check:", archive)
assert manifest["images"] == 1744, manifest["images"]
assert manifest["grade_counts"] == [1017, 270, 347, 75, 35], manifest["grade_counts"]
assert manifest["exams"] == 874, manifest["exams"]
if archive:
    assert archive["archived_images_compared"] == archive["pixel_identical"] == 1057, archive
else:
    print("No archive comparison (the archived 512 px export was not found); the counts above still match.")
print("\nCohort verified.")

## 5. Pilot (about 15 minutes)

One job of each kind on 12 images, written to `final_02__revision_pilot_01` and `retizero_02__revision_pilot_01`. It must print **Pilot passed**: the timing pilot has to reproduce its run's saved predictions exactly. Skip this cell on a resume if the pilot already passed.

In [ ]:
assert run_step("pilot") == 0, "Pilot FAILED: read the log above before the full run."

## 6. Full queue (about 9 hours)

Order: RetiZero Messidor-2 (7), Qwen Messidor-2 (14), then Qwen timing (7). GPU power is sampled every 5 s throughout.

- Finished jobs are skipped, so **rerun this cell to resume** after an interruption or a restart.
- A failed job is recorded and the queue carries on. Rerun the cell to retry it.
- The runner exits with a nonzero code if any job did not pass.

In [ ]:
code = run_step("run")
print("All jobs passed." if code == 0 else "Some jobs did not pass: see the list above, then rerun this cell to retry them.")

## 7. Status and summary

Lists every job. Then it writes `final_02\revision_energy_per_job.csv` and rebuilds the analysis tables in `finetuning\paper_tables\output\revision_analysis\` with the Messidor-2 grades. This step is CPU only and can be rerun any time.

In [ ]:
run_step("status")
assert run_step("summarize", "--messidor-package", MESSIDOR_PACKAGE) == 0, "Summarize failed; see the log above."

## 8. Verify the results

Reads files only. For every job it checks:

- the job is done
- the predictions have one row per image
- the timing jobs agree with their run's saved predictions

It also checks that no file which existed before cell 2 was changed or deleted.

In [ ]:
import pandas as pd

runner = load_runner()
jobs = runner.build_jobs()
rows, issues = [], []
for queue, job in jobs:
    state = job.status()
    predictions = job.output / "predictions.csv"
    n = len(pd.read_csv(predictions)) if predictions.is_file() else 0
    expected = 361 if job.stage == "timing" else 1744
    mismatches = None
    if job.stage == "timing" and (job.output / "summary.json").is_file():
        summary = json.loads((job.output / "summary.json").read_text(encoding="utf-8"))
        mismatches = summary.get("agreement_with_saved_predictions", {}).get("grade_mismatches")
    rows.append({"queue": queue, "stage": job.stage, "job": job.name, "state": state, "images": n, "timing_mismatches": mismatches})
    if state != "done":
        issues.append(f"{job.name}: {state}")
    elif n != expected:
        issues.append(f"{job.name}: {n} predictions, expected {expected}")
    if mismatches:
        issues.append(f"{job.name}: {mismatches} grades differ from the run's saved predictions")
table = pd.DataFrame(rows)
print(table.to_string(index=False))

before = json.loads(SNAPSHOT.read_text(encoding="utf-8"))
changed = []
for path, (size, mtime) in before.items():
    p = Path(path)
    if not p.is_file():
        changed.append(f"deleted: {path}")
    elif (p.stat().st_size, p.stat().st_mtime_ns) != (size, mtime):
        changed.append(f"changed: {path}")
print(f"\nEarlier results: {len(before):,} files checked, {len(changed)} changed or deleted.")
for line in changed[:20]:
    print("  ", line)
issues += changed

print("\nAll 28 jobs verified and earlier results untouched." if not issues and len(jobs) == 28 else
      "\nProblems:\n- " + "\n- ".join(issues[:40]))

## 9. Package the results for the team

Zips what the analysis needs and no model weights:

- both `scores` folders
- the queue state
- the energy file
- the power logs
- this notebook's logs
- the cohort manifest

The zip goes in `D:\Afterimage_data_share\`.

In [ ]:
import zipfile

stamp = datetime.now().strftime("%Y%m%d")
target = Path(r"D:\Afterimage_data_share") / f"revision_gpu_results_{stamp}.zip"
retizero_02 = EXPERIMENTS / "retizero_runs" / "retizero_02"
members = [*(FINAL_02 / "scores").glob("*__messidor2_*/**/*"),
           *(p for p in (FINAL_02 / "scores").glob("timing_*/**/*")),
           *(retizero_02 / "scores").rglob("*"),
           FINAL_02 / "revision_queue_state.json", FINAL_02 / "revision_energy_per_job.csv",
           *FINAL_02.glob("power_log_*.csv"), *LOGS.glob("*"),
           EXPERIMENTS / "datasets/messidor2_verified/manifest.json"]
files = sorted({p for p in members if p.is_file() and "cache" not in p.parts})
with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(EXPERIMENTS))
print(f"{len(files)} files, {target.stat().st_size / 2**20:.1f} MiB -> {target}")

## 10. After the run

1. Commit the rebuilt tables in `finetuning/paper_tables/output/revision_analysis` on this branch.
2. Update the Messidor-2 and timing sections of `docs/paper/REVISION_EVIDENCE.md` from those tables.
3. Send the zip from step 9 to the team.